В рамках данного домашнего задания вам предлагается поработать как с VAE, так и с диффузионными моделями. Для выполения данного задания будем пользоваться экосистемой HuggingFace (конкретно библиотекой для работы с диффузионными моделями diffusers).

За домашнюю работу можно получить до 5 баллов.

# Подготовка окружения

In [ ]:
%%capture
!pip install diffusers==1.3.0
!pip install diffusers==0.32.2
!pip install datasets==3.3.2
!pip install ipywidgets tqdm

In [ ]:
import os
import cv2
from PIL import Image
from IPython.display import display

import torch
import torch.nn.functional as F
import torchvision

import numpy as np

import datasets
import diffusers
import accelerate

from tqdm.auto import tqdm

In [ ]:
device = torch.device("cuda:0")
dtype = torch.float32

# VAE & VQ-VAE (0.5 балла)

В данном задании вам будет необходимо сравнить между собой реконструкции VAE и VQ-VAE на одних и тех же входах.

Подгрузим модели, необходимые нам для работы

In [ ]:
from diffusers import AutoencoderKL, VQModel

# VAE
vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse"
).eval().to(device, dtype)

# VQ-VAE
vqvae = VQModel.from_pretrained(
    "microsoft/vq-diffusion-ithq",
    subfolder="vqvae",
).eval().to(device, dtype)

Для того чтобы моделью можно было обработать входное изображение, необходимо, чтобы каждая сторона делилась нацело на 2 то количество раз, сколько в нашей модели блоков, уменьшающих разрешение в 2 раза.

In [ ]:
# Количество пикселей, на сколько должна делиться каждая сторона входного тензора в VAE
2 ** (len(vae.config.block_out_channels) - 1)

In [ ]:
# Количество пикселей, на сколько должна делиться каждая сторона входного тензора в VQ-VAE
2 ** (len(vqvae.config.block_out_channels) - 1)

In [ ]:
@torch.inference_mode()
def img2tensor(img: np.ndarray) -> torch.Tensor:
    """
    Осуществляет преобразование:
    входного numpy изображения в формате RGB, HWC, диапазон значений (0, 255)
    в выходной torch.Tensor в формате BCHW с диапазоном значений (-1, 1)
    """

    return torch.from_numpy(
        np.transpose(
            (np.asarray(img).astype(np.float32) / 127.5 - 1.0),
            (2, 0, 1)
        )[np.newaxis, ...]
    )

@torch.inference_mode()
def tensor2img(tensor: torch.Tensor) -> np.ndarray:
    """
    Осуществляет преобразование:
    входного torch.Tensor в формате BCHW с диапазоном значений (-1, 1)
    в выходной numpy array в формате BHWC с диапазоном значений (0, 255)
    """

    return (127.5 * (tensor + 1.0)).clamp(0, 255).permute(0, 2, 3, 1).cpu().to(torch.uint8).numpy()

@torch.inference_mode()
def run_vae(x: torch.Tensor, sample: bool = False) -> torch.Tensor:
    """Осуществляет кодирование и декодирование входного тензора с помощью VAE"""

    posterior = vae.encode(x.to(device, dtype)).latent_dist
    # Возьмем моду распределения, или будем осуществлять сэмплирование
    if sample:
        latent = posterior.sample()
    else:
        latent = posterior.mode()
    reconstruction = vae.decode(latent).sample
    return reconstruction

@torch.inference_mode()
def run_vqvae(x: torch.Tensor) -> torch.Tensor:
    """Осуществляет кодирование и декодирование входного тензора с помощью VQ-VAE"""

    latent = vqvae.encode(x.to(device, dtype)).latents
    reconstruction = vqvae.decode(latent).sample
    return reconstruction


def read_image(image_path: str, max_size: int = 512, divisor: int = 8):
    # Read the image
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError("Image not found or cannot be read.")
    
    # Convert to RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Get original dimensions
    height, width = image.shape[:2]
    
    # Determine scaling factor
    scale = max_size / max(height, width)
    new_width = int(width * scale)
    new_height = int(height * scale)
    
    # Ensure new dimensions are divisible by the divisor
    new_width = (new_width // divisor) * divisor
    new_height = (new_height // divisor) * divisor
    
    # Resize the image
    resized_image = cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_AREA)
    
    return resized_image

## Задание 1 (сравнение VAE и VQ-VAE, 0.5 балла)

Возьмите 2-3 любых изображения с минимальной стороной >= 512 пикселей (воспользуйтесь функцией `read_image`). Осуществите сжатие и расжатие с помощью VAE и VQ-VAE и представьте результаты в виде горизонтальной конкатенации 3 изображений - оригинал, реконструкция VAE, реконструкция VQ-VAE.

Повторите данную процедуру для разных входных разрешений (параметр `max_size` в `read_image`).

Сделайте выводы, где на ваш взгляд, результаты реконструкций выглядят лучше, и как зависит качество реконструкций от входного разрешения изображений (на выводы и примеры можно не скупиться).

In [ ]:
#TODO: Ваш код здесь

# Diffusion training & Sampling (4.5 балла)

В данном задании мы будем обучать диффузионную модель на MNIST с помощью diffusers. Обучение будет происходить в pixel space, а не в латентах VAE или VQ-VAE.

Входное разрешение MNIST - 28x28 пикселей, для простоты работы с моделью осуществим ресайз до разрешения 32x32.

Конфигурация обучения

In [ ]:
from dataclasses import dataclass

@dataclass
class TrainingConfig:
    image_size=32 # Resize the digits to be a power of two
    train_batch_size = 32
    eval_batch_size = 32
    num_epochs = 5
    gradient_accumulation_steps = 1
    learning_rate = 1e-4
    lr_warmpup_steps = 500
    mixed_precision = 'fp16'
    seed = 0

config = TrainingConfig()

## Подготовка датасета MNIST

In [ ]:
mnist_dataset = datasets.load_dataset('mnist', split='train')

In [ ]:
mnist_dataset

In [ ]:
mnist_dataset[0]["image"].resize((256, 256)).show()
print("Image Size:", mnist_dataset[0]["image"].size)
print("Digit is labelled:", mnist_dataset[0]['label'])

In [ ]:
def transform(dataset):
    preprocess = torchvision.transforms.Compose(
        [
            torchvision.transforms.Resize(
                (config.image_size, config.image_size)),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Lambda(lambda x: 2*(x-0.5)),
        ]
    )
    images = [preprocess(image) for image in dataset["image"]]
    return {"images": images}

In [ ]:
mnist_dataset.reset_format()
mnist_dataset.set_transform(transform)

In [ ]:
train_dataloader = torch.utils.data.DataLoader(
    mnist_dataset,
    batch_size = config.train_batch_size,
    shuffle = True,
)

## Подготовка диффузионной модели

Будем обучать модель без условия, чтобы она генерировала нам случайную цифру.

In [ ]:
from diffusers import UNet2DModel

model = UNet2DModel(
    sample_size=config.image_size,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(128,128,256,512),
    down_block_types=(
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D",
        "DownBlock2D",
    ),
    up_block_types=(
        "UpBlock2D",
        "AttnUpBlock2D",
        "UpBlock2D",
        "UpBlock2D",
    ),
)

Проверим, что наша модель выдает корректное выходное разрешение.

In [ ]:
sample_image = mnist_dataset[0]["images"].unsqueeze(0)
print("Input shape:", sample_image.shape)

In [ ]:
print('Output shape:', model(sample_image, timestep=0)["sample"].shape)

## Расписание шума

Воспользуемся линейным расписанием шума из статьи [DDPM](https://arxiv.org/abs/2006.11239).

Поскольку наш датасет простой, снизим количество диффузионных шагов с 1000 до 200. Так процесс сэмплирования будет работать быстрее.

In [ ]:
from diffusers import DDPMScheduler

noise_scheduler = diffusers.DDPMScheduler(
    num_train_timesteps=200,
    beta_schedule="linear"
)

Зашумим случайный пример

In [ ]:
print("Original Digit")
torchvision.transforms.ToPILImage()(0.5 * (sample_image.squeeze(1) + 1.0)).resize((256,256))

In [ ]:
noise = torch.randn(sample_image.shape)
timesteps = torch.LongTensor([199])
noisy_image = noise_scheduler.add_noise(sample_image,noise,timesteps)

print("Fully Noised Digit")
torchvision.transforms.ToPILImage()(
    0.5 * (noisy_image.squeeze(1) + 1.0)
).resize((256,256))

## Оптимизатор

Воспользуемся оптимизатором AdamW.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(),lr=config.learning_rate)

## Learning rate scheduler

В статье [Improved Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2102.09672) было замечено, что добавление learning rate scheduler'а, который сперва осуществляет прогрев, а затем меняет learning rate с помощью cosine scheduler делает сходимость лучше. Воспользуемся готовыми методами в diffusers

In [ ]:
# Cosine learning rate scheduler

lr_scheduler = diffusers.optimization.get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=config.lr_warmpup_steps,
    num_training_steps=(len(train_dataloader)*config.num_epochs),
)

## Обучение модели

Опишем функцию, которая будет осуществлять обучение модели. Перед этим подготовим модель с помощью `accelerate`, чтобы она могла эффективно учиться в нужном нам типе данных и с потенциальной аккумуляцией градиентов между несколькими оптимизациями.

In [ ]:
def train_loop(
        config,
        model,
        noise_scheduler,
        optimizer,
        train_dataloader,
        lr_scheduler):

    accelerator = accelerate.Accelerator(
        mixed_precision=config.mixed_precision,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
    )

    model, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
        model, optimizer, train_dataloader, lr_scheduler
    )

    for epoch in range(config.num_epochs):
        progress_bar = tqdm(total=len(train_dataloader),
                            disable=not accelerator.is_local_main_process)
        progress_bar.set_description(f"Epoch {epoch}")

        for step, batch in enumerate(train_dataloader):
            clean_images = batch['images']

            noise = torch.randn(clean_images.shape).to(clean_images.device)
            batch_size = clean_images.shape[0]

            # Sample a set of random time steps for each image in mini-batch
            timesteps = torch.randint(
                0, noise_scheduler.num_train_timesteps, (batch_size,), device=clean_images.device)

            noisy_images=noise_scheduler.add_noise(clean_images, noise, timesteps)

            with accelerator.accumulate(model):
                noise_pred = model(noisy_images,timesteps)["sample"]
                loss = torch.nn.functional.mse_loss(noise_pred,noise)
                accelerator.backward(loss)

                accelerator.clip_grad_norm_(model.parameters(),1.0)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            progress_bar.update(1)
            logs = {
                "loss" : loss.detach().item(),
                "lr" : lr_scheduler.get_last_lr()[0],
            }
            progress_bar.set_postfix(**logs)

    accelerator.unwrap_model(model)

Запустим обучение диффузионной модели (должно занимать до 20-30 минут на NVIDIA карточке).

In [ ]:
args = (config, model, noise_scheduler, optimizer, train_dataloader, lr_scheduler)

accelerate.notebook_launcher(train_loop, args, num_processes=1)

## Сэмплирование из обученной модели

Реализуем вспомогательную функцию для осуществления сэмплирования из обученной модели.

In [ ]:
@torch.no_grad()
def sample(unet, scheduler,seed,save_process_dir=None):
    torch.manual_seed(seed)

    if save_process_dir:
        if not os.path.exists(save_process_dir):
            os.mkdir(save_process_dir)

    scheduler.set_timesteps(200)
    image=torch.randn((1,1,32,32)).to(model.device)
    num_steps=max(noise_scheduler.timesteps).numpy()

    for t in tqdm(noise_scheduler.timesteps):
        model_output=unet(image,t)['sample']
        image=scheduler.step(model_output,int(t),image,generator=None)['prev_sample']
        if save_process_dir:
            save_image=torchvision.transforms.ToPILImage()(
                0.5 * (image.squeeze(0) + 1.0)
            )
            save_image.resize((256,256)).save(
                os.path.join(save_process_dir,"seed-"+str(seed)+"_"+f"{num_steps-t.numpy():03d}"+".png"),format="png")

    return torchvision.transforms.ToPILImage()(
        0.5 * (image.squeeze(0) + 1.0)
    )

Видим, что не все генерации похожи на цифры.

In [ ]:
test_image=sample(model,noise_scheduler,2)
test_image.resize((265,256))

Но при этом присутствуют и хорошие генерации.

In [ ]:
test_image=sample(model,noise_scheduler,42)
test_image.resize((265,256))

In [ ]:
test_image=sample(model,noise_scheduler,1991)
test_image.resize((265,256))

## Задание 2 (inpainting без переобучения модели, 1.5 балла)

Скопируйте и модифицируйте функцию, осуществляющую сэмплирование из обученной диффузионной модели выше. Модифицируйте ее таким образом, чтобы она решала задачу inpainting (восстанавливала замаскированную часть входного изображения).

Новая функция сэмплирования должна дополнительно принимать замаскированный пример и бинарную маску. Для простоты сконструируйте 4 маски, закрывающие верхнюю, нижнюю, левую и правую часть входного примера.

Продемонстрируйте результаты генерации для 3-4 входных примеров и 3-4 разных входных сидов.
Оцените, как работает модель на одних и тех же входных примерах, но разных сидах, сделайте выводы.

In [ ]:
#TODO: Ваш код здесь

## Задание 3 (inpainting с переобучением модели, 3 балла)

Скопируйте и модифицируйте код обучения диффузионной модели выше таким образом, чтобы вы обучали диффузионную модель специально для inpainting.

Для этого модель теперь должна принимать 6 каналов (3 канала для текущего зашумленного изображения, 3 канала для замаскированного примера, 1 канал для маски), в модель необходимо подавать конкатенацию всех трех тензоров по размерности каналов. Как и раньше, модель должна выдавать предсказанный шум.

Для простоты переиспользуйте 4 маски из задания 1 (маски, закрывающие верхнюю, нижнюю, левую и правую части изображения).

При обучении в 1 батч данных лучше подавать разные маски, чтобы на каждой итерации модель училась работать с разными масками.

Напишите соответствующую данной модели функцию сэмплирования и продемонстрируйте ее работу на 3-4 входных примерах и 3-4 разных входных сидов. Оцените, как работает модель на одних и тех же входных примерах, но на разных сидах. Сравните, насколько данная модель лучше справляется с задачей inpainting по сравнению с подходом из задания 1.

In [ ]:
#TODO: Ваш код здесь